In [2]:
import pandas as pd
import requests
import zipfile
from io import BytesIO
from pyspark.sql import functions as F

# Download ECB ZIP file
url = "https://www.ecb.europa.eu/stats/eurofxref/eurofxref-hist.zip"

response = requests.get(url, timeout=60)
response.raise_for_status()

# Extract CSV from ZIP
with zipfile.ZipFile(BytesIO(response.content)) as z:
    csv_files = [f for f in z.namelist() if f.endswith(".csv")]

    if not csv_files:
        raise Exception("No CSV file found inside ECB ZIP!")

    with z.open(csv_files[0]) as csv_file:
        fx_pd = pd.read_csv(csv_file)

print("Downloaded ECB records:", len(fx_pd))
print("Available columns:", fx_pd.columns.tolist())

# Select Date and USD
fx_pd = fx_pd[["Date", "USD"]].copy()

fx_pd["Date"] = pd.to_datetime(fx_pd["Date"])
fx_pd["USD"] = pd.to_numeric(fx_pd["USD"], errors="coerce")

fx_pd = fx_pd.dropna(subset=["Date", "USD"])

# Convert to Spark DataFrame
fx_df = spark.createDataFrame(fx_pd)

# Save to Bronze
(
    fx_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ecb_usd_eur_raw")
)

print("ECB data saved successfully!")
print("Total records:", fx_df.count())

display(fx_df.orderBy(F.desc("Date")).limit(10))

StatementMeta(, 4a880cab-8912-4a2b-8726-48ac84d12e10, 4, Finished, Available, Finished, False)

Downloaded ECB records: 7127
Available columns: ['Date', 'USD', 'JPY', 'BGN', 'CYP', 'CZK', 'DKK', 'EEK', 'GBP', 'HUF', 'LTL', 'LVL', 'MTL', 'PLN', 'ROL', 'RON', 'SEK', 'SIT', 'SKK', 'CHF', 'ISK', 'NOK', 'HRK', 'RUB', 'TRL', 'TRY', 'AUD', 'BRL', 'CAD', 'CNY', 'HKD', 'IDR', 'ILS', 'INR', 'KRW', 'MXN', 'MYR', 'NZD', 'PHP', 'SGD', 'THB', 'ZAR', 'Unnamed: 42']
ECB data saved successfully!
Total records: 7127


SynapseWidget(Synapse.DataFrame, fb40ce70-690e-486b-b6e8-1e8efad6e6a5)